In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd drive/MyDrive/Colab_project/model_trained_MNIST/ResNet_for_MNIST


In [ ]:
!pip install pytorch-lightning wandb matplotlib scipy
!pip install torchvision tqdm

In [ ]:
%%writefile training_fixed_ResNet_224.py
import numpy as np
import pandas as pd
import random
import argparse
import time

import pytorch_lightning as pl
import torch
from torch import nn, optim
from torchmetrics import Accuracy
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger

import multiprocessing as mp

try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import os
os.environ['PYTHONWARNINGS'] = "ignore"


def set_random_seed(inst):
    np.random.seed(42)

    random_seeds = np.random.choice(range(1, 10000), size=100, replace=False)
    inst_seed = int(random_seeds[inst])
    print(f"Using seed: {inst_seed} for Instance {inst}")

    np.random.seed(inst_seed)
    random.seed(inst_seed)
    torch.manual_seed(inst_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(inst_seed)
    pl.seed_everything(inst_seed)

def load_dataset():

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = datasets.MNIST(root='data', train=True, transform=train_transform, download=True)
    test_dataset = datasets.MNIST(root='data', train=False, transform=test_transform, download=True)


    train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, test_loader


class LitResNet18(pl.LightningModule):
    def __init__(self, num_classes=10, lr=1e-4):
        super(LitResNet18, self).__init__()
        self.save_hyperparameters()

        self.model = models.resnet18(weights=None)

        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)


        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

        self.accuracy = Accuracy(num_classes=num_classes, task='multiclass')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        acc = self.accuracy(outputs.softmax(dim=-1), labels)

        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        acc = self.accuracy(outputs.softmax(dim=-1), labels)

        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        self.log('val_acc', acc, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def configure_optimizers(self):

        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)

        # --- FIX: Removed verbose=True ---
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=2
        )

        lr_scheduler_config = {
            'scheduler': scheduler,
            'monitor': 'val_loss',
            'interval': 'epoch',
            'frequency': 1
        }
        return [optimizer], [lr_scheduler_config]


def main():
    parser = argparse.ArgumentParser(description='Train ResNet18 on MNIST (224x224)')
    parser.add_argument('--instance', type=int, required=True, help='Instance number (0-59)')
    parser.add_argument('--epochs', type=int, default=20, help='Number of epochs')
    args = parser.parse_args()

    set_random_seed(args.instance)

    # Logger
    wandb_logger = WandbLogger(project='ResNet18_MNIST_224', name=f"resnet_inst_{args.instance}")

    # Callbacks
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    early_stop_callback = EarlyStopping(
        monitor='val_loss', min_delta=0.0005, patience=5, verbose=True, mode='min'
    )
    checkpoint_callback = ModelCheckpoint(
        monitor='val_acc',
        dirpath='model_resnet_224/',
        filename=f'resnet-224-{args.instance}',
        save_top_k=1,
        mode='max'
    )

    # Initialize Model
    model = LitResNet18(num_classes=10)

    # Trainer
    trainer = pl.Trainer(
        max_epochs=args.epochs,
        accelerator="auto",
        devices=1,
        callbacks=[checkpoint_callback, lr_monitor, early_stop_callback],
        logger=wandb_logger,
        log_every_n_steps=50
    )

    train_loader, test_loader = load_dataset()

    print(f"Starting training for Instance {args.instance} with 224x224 input...")
    trainer.fit(model, train_loader, test_loader)

    # Save final state for consistency with your other scripts
    torch.save(model.state_dict(), f'model_resnet_224/resnet-224-{args.instance}-final.pt')
    print("Training Complete.")

if __name__ == "__main__":
    main()

In [ ]:
!ls -la

# 5. Install or update PyTorch with CUDA support
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

# 6. Create data directory if it doesn't exist
!mkdir -p data

# 7. Download MNIST dataset (modify train_model.py has download=False)
import torchvision
torchvision.datasets.MNIST(root='data', train=True, download=True)
torchvision.datasets.MNIST(root='data', train=False, download=True)


In [ ]:
import wandb
wandb.login()


In [ ]:
def train_instance(instance_number):
    print(f"\n{'='*50}")
    print(f"Training Instance {instance_number}/60")
    print(f"{'='*50}\n")

    !python training_fixed_ResNet_224.py --instance {instance_number}

    print(f"\nCompleted Instance {instance_number}/60")
    return f"Instance {instance_number} completed"


In [ ]:
start_instance = 0  # Change this if resuming from a previous run
end_instance = 59  # Inclusive end

In [ ]:
from google.colab import files
import time
start_time = time.time()

# Run training loop with improved tracking
for i in range(start_instance, end_instance + 1):
    try:
        result = train_instance(i)
        print(result)

        # Progress tracking
        elapsed_time = time.time() - start_time
        instances_completed = i - start_instance + 1
        total_instances = end_instance - start_instance + 1

        avg_time_per_instance = elapsed_time / instances_completed
        estimated_remaining = avg_time_per_instance * (total_instances - instances_completed)

        print(f"\nProgress: {instances_completed}/{total_instances} instances completed")
        print(f"Elapsed time: {elapsed_time/60:.2f} minutes")
        print(f"Estimated remaining time: {estimated_remaining/60:.2f} minutes")

    except Exception as e:
        print(f"Error in instance {i}: {e}")

    # Clear GPU memory between runs
    import gc
    gc.collect()
    torch.cuda.empty_cache()

# Final summary
print("\n=== Training Complete ===")
print(f"Total time: {(time.time() - start_time)/60:.2f} minutes")